# Petunjuk menjalankan notebook yang dilengkapi

Sel awal tim tetap utuh sebagai alur sebelumnya. Untuk menjalankan delapan eksperimen dengan pembagian data dan metrik seragam, buka bagian **Tambahan eksperimen lengkap** setelah sel awal, lalu jalankan mulai dari sel setup bagian tersebut sampai akhir. Setiap langkah tambahan menjelaskan tujuan kode dan cara membaca hasilnya. Script `python src/execute_notebooks.py` menjalankan bagian tambahan secara otomatis.

In [3]:
import pandas as pd
import numpy as np

In [4]:
df_raw = pd.read_csv('D:/MK5 Project Sistem Cerdas/messy-food-waste-prediction-dataset/train.csv')
if 'date' in df_raw.columns:
    df_raw['date'] = pd.to_datetime(df_raw['date'])
    df_raw = df_raw.sort_values('date')

In [5]:
split_index = int(len(df_raw) * 0.8)
train_raw = df_raw.iloc[:split_index].copy()
test_raw = df_raw.iloc[split_index:].copy()

print(f"Data dipecah: Train ({len(train_raw)} baris), Test ({len(test_raw)} baris)")

Data dipecah: Train (728 baris), Test (183 baris)


In [6]:
# EKSPERIMEN A: STANDARD PREPROCESSING

train_std = train_raw.copy().drop(columns=['date'], errors='ignore')
test_std = test_raw.copy().drop(columns=['date'], errors='ignore')

# Imputasi Sederhana
num_cols = train_std.select_dtypes(include=['number']).columns.drop('food_waste_kg', errors='ignore')
cat_cols = train_std.select_dtypes(include=['object']).columns

for col in num_cols:
    train_std[col] = train_std[col].fillna(train_std[col].median())
    test_std[col] = test_std[col].fillna(train_std[col].median()) # Pakai median dari train
for col in cat_cols:
    train_std[col] = train_std[col].fillna(train_std[col].mode()[0])
    test_std[col] = test_std[col].fillna(train_std[col].mode()[0])

train_std.to_csv('catboost_train_standard.csv', index=False)
test_std.to_csv('catboost_test_standard.csv', index=False)

C:\Users\davew\AppData\Local\Temp\ipykernel_30616\3717779426.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_std.select_dtypes(include=['object']).columns


In [7]:
# EKSPERIMEN B: ROBUST PREPROCESSING

train_rob = train_raw.copy().drop_duplicates()
test_rob = test_raw.copy()

# 1. Pembersihan Teks
for col in cat_cols:
    train_rob[col] = train_rob[col].astype(str).str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
    test_rob[col] = test_rob[col].astype(str).str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)

# 2. Imputasi
for col in num_cols:
    train_rob[col] = train_rob[col].fillna(train_rob[col].median())
    test_rob[col] = test_rob[col].fillna(train_rob[col].median())
for col in cat_cols:
    train_rob[col] = train_rob[col].replace('nan', np.nan).fillna(train_rob[col].mode()[0])
    test_rob[col] = test_rob[col].replace('nan', np.nan).fillna(train_rob[col].mode()[0])
    
# 3. Penanganan Outlier (HANYA pada Train data, Test data dibiarkan apa adanya)
for col in num_cols:
    Q1 = train_rob[col].quantile(0.25)
    Q3 = train_rob[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    train_rob[col] = np.clip(train_rob[col], lower, upper)

# 4. Feature Engineering (Lag & Tanggal)
if 'date' in train_raw.columns:
    for df_temp, df_orig in zip([train_rob, test_rob], [train_raw, test_raw]):
        df_temp['hari_dalam_minggu'] = df_orig['date'].dt.dayofweek
        df_temp['is_weekend'] = df_temp['hari_dalam_minggu'].apply(lambda x: 1 if x >= 5 else 0)
        df_temp['limbah_h_minus_1'] = df_orig['food_waste_kg'].shift(1).fillna(df_orig['food_waste_kg'].median())
        df_temp.drop(columns=['date'], inplace=True, errors='ignore')

# Ekspor Data Robust
train_rob.to_csv('catboost_train_robust.csv', index=False)
test_rob.to_csv('catboost_test_robust.csv', index=False)

print("Preprocessing Selesai. Data Standard dan Robust berhasil diekspor!")

Preprocessing Selesai. Data Standard dan Robust berhasil diekspor!


In [8]:
# TAMBAHAN: PERSIAPAN DATA UNTUK MODEL LAIN (LSTM, RF, XGBOOST)
# (Melakukan One-Hot Encoding dan Scaling untuk Standard & Robust)

from sklearn.preprocessing import StandardScaler
import joblib

print("Mulai memproses data untuk model teman-teman...")

def siapkan_data_others(df_train, df_test, prefix_nama):
    # 1. One-Hot Encoding (Mengubah teks kategori jadi angka 0 dan 1)
    cat_cols_all = df_train.select_dtypes(include=['object']).columns
    train_encoded = pd.get_dummies(df_train, columns=cat_cols_all, drop_first=True)
    test_encoded = pd.get_dummies(df_test, columns=cat_cols_all, drop_first=True)
    
    # Menyamakan jumlah kolom antara train dan test
    test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0)
    
    X_train = train_encoded.drop(columns=['food_waste_kg'], errors='ignore')
    y_train = train_encoded['food_waste_kg']
    X_test = test_encoded.drop(columns=['food_waste_kg'], errors='ignore')
    y_test = test_encoded['food_waste_kg']
    
    # 2. Scaling (Menyamakan skala angka)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)
    
    # 3. Ekspor Data
    X_train_scaled.to_csv(f'others_train_{prefix_nama}_features.csv', index=False)
    y_train.to_csv(f'others_train_{prefix_nama}_target.csv', index=False)
    X_test_scaled.to_csv(f'others_test_{prefix_nama}_features.csv', index=False)
    y_test.to_csv(f'others_test_{prefix_nama}_target.csv', index=False)
    
    # Simpan scaler untuk keperluan Streamlit nanti
    joblib.dump(scaler, f'scaler_{prefix_nama}.pkl')
    print(f"✅ Data '{prefix_nama}' untuk LSTM/RF/XGBoost berhasil diekspor!")

# Eksekusi fungsi untuk kedua versi data
siapkan_data_others(train_std, test_std, "standard")
siapkan_data_others(train_rob, test_rob, "robust")

Mulai memproses data untuk model teman-teman...
✅ Data 'standard' untuk LSTM/RF/XGBoost berhasil diekspor!
✅ Data 'robust' untuk LSTM/RF/XGBoost berhasil diekspor!


C:\Users\davew\AppData\Local\Temp\ipykernel_30616\3795820832.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_all = df_train.select_dtypes(include=['object']).columns
C:\Users\davew\AppData\Local\Temp\ipykernel_30616\3795820832.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/doc

## Tambahan eksperimen lengkap

Kode awal milik tim di atas tetap dipertahankan tanpa penghapusan. Bagian di bawah menyediakan alur tambahan yang dapat dijalankan mulai dari sel setup hingga akhir, tanpa menjalankan ulang sel lama. Hasil alur lama tidak dicampur karena pembagian data dan asumsi fiturnya berbeda. Script eksekusi otomatis melewati sel referensi lama bertag `foodsave-legacy`.

# 03 Menyiapkan data Standard dan Robust

**Tujuan:** membandingkan dua cara membersihkan data pada pembagian waktu yang sama. Training memakai sekitar 60% tanggal awal, validasi 20% berikutnya, dan test 20% terakhir. Persentase jumlah baris dapat berbeda karena tanggal tidak dipisah.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / 'src/experiment.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src/experiment.py').exists(), 'Buka notebook dari folder proyek FoodSave.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown, Code
from src.experiment import OUT, ART, load_data, prepare, run_model, evaluate
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)
get_ipython().run_line_magic('matplotlib', 'inline')


## Perbedaan pendekatan

| Langkah | Standard | Robust |
|---|---|---|
| Konversi angka | Konversi sederhana | Bersihkan simbol, koma pemisah ribuan, dan spasi |
| Kategori | Pertahankan penulisan | Huruf kecil dan pangkas spasi |
| Nilai kosong | Median/modus training | Median/modus training |
| Nilai ekstrem fitur | Dipertahankan | Batasi dengan IQR training |
| Skala LSTM | StandardScaler | RobustScaler |
| Target aktual | Tidak diubah | Tidak diubah |

Duplikat persis dihapus untuk kedua pendekatan sebelum split demi menjaga identitas observasi. Batas IQR yang dipelajari dari training diterapkan juga ke validasi/test; **target validasi/test tidak dipotong**. Model pohon tidak memerlukan scaling. CatBoost menerima kategori asli setelah pembersihan; model lainnya memakai one-hot encoding.

In [2]:
audit_rows=[]
for method in ['standard','robust']:
    df, prep, idx, ys, x, xs, native=prepare(method)
    assert np.isfinite(x.to_numpy()).all() and np.isfinite(xs).all()
    assert not set(['ID','waste_category','past_waste_kg','food_waste_kg']).intersection(x.columns)
    for split, ix in idx.items():
        folder=ROOT/'dataset/processed'; folder.mkdir(parents=True,exist_ok=True)
        x.iloc[ix].to_csv(folder/f'validated_{method}_{split}_features.csv',index=False)
        df.iloc[ix][['_row','date','food_waste_kg']].to_csv(folder/f'validated_{method}_{split}_target.csv',index=False)
    audit_rows.append({'metode':method,'fitur':x.shape[1], **{s:len(ix) for s,ix in idx.items()},'missing_setelah':int(x.isna().sum().sum())})
    display(Markdown(f'### Contoh hasil {method}')); display(native.head())
    display(prep.bounds)
display(pd.DataFrame(audit_rows))

### Contoh hasil standard

,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,Intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
4,370.0,14.0,30.4987,64.6350,0.0,Beginner,0.0,0.5,0.866,0.0378,26.4286,0.0


{}

### Contoh hasil robust

,meals_served,kitchen_staff,temperature_C,humidity_percent,special_event,staff_experience,is_weekend,month_sin,month_cos,staff_per_portion,portion_per_staff,special_event_portion
0,202.0,9.0,31.9768,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
1,202.0,9.0,32.3188,66.8266,0.0,intermediate,1.0,0.5,0.866,0.0446,22.4444,0.0
2,448.0,16.0,10.0925,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
3,448.0,16.0,10.1286,56.1303,0.0,intermediate,1.0,0.5,0.866,0.0357,28.0000,0.0
4,370.0,14.0,30.4987,64.6350,0.0,beginner,0.0,0.5,0.866,0.0378,26.4286,0.0


{'meals_served': (-83.5, 704.5),
 'kitchen_staff': (-2.5, 25.5),
 'temperature_C': (-3.6113231948656654, 48.165288450172554),
 'humidity_percent': (1.2913035387851082, 120.67113368570207)}

,metode,fitur,train,validation,test,missing_setelah
0,standard,15,558,173,174,0
1,robust,14,558,173,174,0


## Memahami fitur turunan dan sequence
`staff_per_portion` adalah staf dibagi jumlah porsi, `portion_per_staff` adalah porsi per staf, dan `special_event_portion` adalah interaksi acara dengan porsi. Bulan diubah ke sinus/kosinus agar Desember dekat dengan Januari.

LSTM memakai jendela **7 observasi**, termasuk input operasional hari yang hendak direncanakan. Ini bukan jaminan 7 hari berurutan. Tidak ada target aktual yang dimasukkan ke jendela. Konteks validasi/test boleh berisi fitur observasi sebelumnya, tanpa mengambil label atau baris masa depan. Enam baris awal dihilangkan dari evaluasi semua model agar perbandingannya setara.

In [3]:
import inspect
from src.experiment import FoodPreprocessor, sequence
display(Code(inspect.getsource(FoodPreprocessor),language='python'))
display(Code(inspect.getsource(sequence),language='python'))

class FoodPreprocessor:
    """Standard: konversi sederhana. Robust: normalisasi dan batas IQR training."""
    def __init__(self, method='standard'):
        self.method = method

    def clean(self, df):
        d = df[NUM + CAT].copy()
        for c in NUM:
            s = d[c]
            if self.method == 'robust':
                s = s.astype('string').str.strip().str.replace(',', '', regex=False).str.replace(r'[^0-9.eE+\-]', '', regex=True)
            # Float mendukung median/batas IQR pecahan, termasuk pada pandas 3.
            d[c] = pd.to_numeric(s, errors='coerce').astype('float64').replace([np.inf, -np.inf], np.nan)
        for c in CAT:
            s = d[c].astype('string')
            if self.method == 'robust':
                s = s.str.strip().str.lower().replace({'nan': pd.NA, 'none': pd.NA, '': pd.NA})
            d[c] = s
        date = pd.to_datetime(df['date'])
        d['is_weekend'] = (date.dt.dayofweek >= 5).astype(float)
        d['month_sin'] = np.sin(2 * np.pi * date.dt.month / 12)
        d['month_cos'] = np.cos(2 * np.pi * date.dt.month / 12)
        return d

    def fit(self, df):
        d = self.clean(df)
        self.numerics = d.select_dtypes(include='number').columns.tolist()
        self.medians = d[self.numerics].median().fillna(0)
        self.modes = {c: (d[c].dropna().mode().iloc[0] if d[c].notna().any() else 'unknown') for c in CAT}
        self.bounds = {}
        if self.method == 'robust':
            for c in ['meals_served', 'kitchen_staff', 'temperature_C', 'humidity_percent']:
                q1, q3 = d[c].quantile([.25, .75]); iqr = q3 - q1
                self.bounds[c] = (float(q1 - 1.5 * iqr), float(q3 + 1.5 * iqr))
        native = self.native(df)
        self.columns = pd.get_dummies(native, columns=CAT, dtype=float).columns.tolist()
        self.scaler = (RobustScaler() if self.method == 'robust' else StandardScaler())
        self.scaler.fit(self.encoded(df))
        return self

    def native(self, df):
        d = self.clean(df)
        d[self.numerics] = d[self.numerics].fillna(self.medians)
        for c, bounds in self.bounds.items():
            d[c] = d[c].clip(*bounds)
        for c in CAT:
            d[c] = d[c].fillna(self.modes[c]).astype(str)
        # Rasio dihitung setelah imputasi dan penanganan outlier.
        d['staff_per_portion'] = d.kitchen_staff / d.meals_served.clip(lower=1)
        d['portion_per_staff'] = d.meals_served / d.kitchen_staff.clip(lower=1)
        d['special_event_portion'] = d.special_event * d.meals_served
        return d

    def encoded(self, df):
        return pd.get_dummies(self.native(df), columns=CAT, dtype=float).reindex(columns=self.columns, fill_value=0).astype(float)

    def transform(self, df, scaled=False):
        x = self.encoded(df)
        return self.scaler.transform(x).astype('float32') if scaled else x

def sequence(x, indices, window=WINDOW):
    """Jendela fitur s.d. hari rencana saat ini; tanpa target aktual sebagai input.

    Fitur operasional hari target diasumsikan sudah diketahui ketika merencanakan.
    Observasi masa depan dan label test tidak pernah dipakai sebagai konteks.
    """
    return np.asarray([x[i-window+1:i+1] for i in indices], dtype='float32')

## Audit tambahan sebelum dan sesudah pembersihan
Tabel berikut menampilkan contoh mentah dan hasil pembersihan tanpa mengganti file processed lama. Kolom `validated_` dipakai untuk keluaran tambahan. Semua input akhir harus finite, urutan fitur sama antarpartisi, dan target tetap sama untuk kedua pendekatan.

In [4]:
df=load_data()
summary=[]
for method in ['standard','robust']:
    prep=FoodPreprocessor(method).fit(df[df.split=='train'])
    cleaned=prep.native(df)
    display(Markdown(f'### {method}'))
    display(pd.concat([df[['meals_served','kitchen_staff','staff_experience']].head(8).add_prefix('asli_'),cleaned[['meals_served','kitchen_staff','staff_experience']].head(8).add_prefix('hasil_')],axis=1))
    transformed=prep.transform(df)
    for split in ['train','validation','test']:
        part=transformed.loc[df.split==split]
        summary.append({'metode':method,'partisi':split,'baris':len(part),'fitur':part.shape[1],'kosong':int(part.isna().sum().sum()),'finite':bool(np.isfinite(part.to_numpy()).all())})
display(pd.DataFrame(summary))


### standard

,asli_meals_served,asli_kitchen_staff,asli_staff_experience,hasil_meals_served,hasil_kitchen_staff,hasil_staff_experience
0,202,9,Intermediate,202.0,9.0,Intermediate
1,202,9,Intermediate,202.0,9.0,Intermediate
2,448,16,intermediate,448.0,16.0,intermediate
3,448,16,intermediate,448.0,16.0,intermediate
4,370,14,Beginner,370.0,14.0,Beginner
5,370,14,Beginner,370.0,14.0,Beginner
6,206,11,Beginner,206.0,11.0,Beginner
7,206,11,Beginner,206.0,11.0,Beginner


### robust

,asli_meals_served,asli_kitchen_staff,asli_staff_experience,hasil_meals_served,hasil_kitchen_staff,hasil_staff_experience
0,202,9,Intermediate,202.0,9.0,intermediate
1,202,9,Intermediate,202.0,9.0,intermediate
2,448,16,intermediate,448.0,16.0,intermediate
3,448,16,intermediate,448.0,16.0,intermediate
4,370,14,Beginner,370.0,14.0,beginner
5,370,14,Beginner,370.0,14.0,beginner
6,206,11,Beginner,206.0,11.0,beginner
7,206,11,Beginner,206.0,11.0,beginner


,metode,partisi,baris,fitur,kosong,finite
0,standard,train,564,15,0,True
1,standard,validation,173,15,0,True
2,standard,test,174,15,0,True
3,robust,train,564,14,0,True
4,robust,validation,173,14,0,True
5,robust,test,174,14,0,True


## Menguji bahwa perubahan target masa depan tidak mengubah fitur
Tes berikut menyalin data lalu mengganti seluruh label validasi/test dengan angka sembarang. Hasil fitur harus tetap sama. Jika berubah, target bocor ke pipeline. Ini pemeriksaan kode; bukan bukti bahwa semua calon fitur pasti tersedia saat operasional.

In [5]:
changed=df.copy();changed.loc[changed.split!='train','food_waste_kg']=9999999
for method in ['standard','robust']:
    prep=FoodPreprocessor(method).fit(df[df.split=='train'])
    pd.testing.assert_frame_equal(prep.transform(df),prep.transform(changed))
    print(method,': LULUS, target masa depan tidak memengaruhi fitur')


standard : LULUS, target masa depan tidak memengaruhi fitur
robust : LULUS, target masa depan tidak memengaruhi fitur
